<a id="inicio"></a>
    # AED 01 · De la decisión al contrato de datos

    **Núcleo U1 · S01–S02 · RA1 / CE1.1–1.2; RA2 / CE2.2** · Colección docente · Edición 2026-10-08.

    Ejecute todas las celdas en orden con el entorno documentado en `../reproducibilidad/README.md`.
    Los controles necesitan kernel activo; las salidas guardadas permiten lectura. Los datos son
    casos docentes locales; las simulaciones y parámetros económicos se identifican como supuestos.
    Consulte la [guía maestra](../guia_maestra_aed.html), el
    [manual científico](../material_propio/AED_manual_cientifico.pdf) y la
    [matriz curricular](../COBERTURA.md). Los ejercicios son formativos.

### Antes de ejecutar

Núcleo · 120 min guiados + 45 min autónomos orientativos.

**Objetivo:** Diseñar un proceso con entregables y reglas de calidad que cambian la decisión.

**Preparación:** Claves, tipos de variable y pregunta de negocio.

**Ejemplo de referencia:** CasaPeumo: comparar original y derivado; BoldoNet: distinguir cohortes maduras de incompletas.

**Práctica:** Mover la fecha de corte y explicar por qué cambia la población elegible del KPI.

**Criterio de logro:** Entregar contrato, auditoría, denominadores y registro de exclusiones reproducible.

**Distribución orientativa:** explicación 30 min; práctica guiada 90 min; trabajo autónomo 45 min. Ajustar tras una clase piloto.

[Guía y secuencia](../guia_maestra_aed.html#ruta-aprendizaje) · [Fundamento del manual](../material_propio/AED_manual_cientifico.html#sec-proceso)

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, Markdown

RAIZ = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "_transversal/datos").is_dir())
sys.path.insert(0, str(RAIZ / "04_Analitica_Estrategica_Datos/reproducibilidad"))
from aed_recursos import (
    casapeumo,
    indicadores,
    boldonet,
    plan_produccion,
    decision,
    solar,
    resumen_solar,
    pregunta,
    DATOS,
    SEMILLA,
)

plt.rcParams.update(
    {"figure.figsize": (8, 3.5), "axes.spines.top": False, "axes.spines.right": False}
)

<a id="proceso"></a>
## 1. Diseñar un proceso que responda una decisión

**Caso CasaPeumo.** Operaciones debe decidir dónde iniciar un piloto para reducir reclamos y
demoras, manteniendo margen. Una venta es la unidad de observación; una región/canal es una
posible unidad de intervención. No son intercambiables. La secuencia es pregunta → fuentes →
contrato y calidad → descripción → alternativas → piloto → seguimiento. CRISP-DM permite
iterar: un problema de cobertura puede devolvernos a la comprensión del negocio.

Ficha mínima: responsable, acción, población, horizonte, KPI, comparación, costo, capacidad y
criterio de suspensión. «Aumentar ventas» no basta: ingreso, margen y experiencia pueden moverse
en direcciones distintas. El proceso debe producir evidencia para escoger una acción y también
permitir concluir que falta evidencia.

**Objetivo del laboratorio:** documentar cada transformación, mantener trazabilidad y distinguir
errores de hechos incómodos. Un resultado extremo legítimo no se elimina porque empeore el KPI.

In [2]:
ventas, tablas, auditoria = casapeumo()
display(pd.Series(auditoria, name="conteo").to_frame())
display(ventas[["IdVenta", "Fecha", "Canal", "Ingreso", "Margen", "Alta_posterior"]].head())
assert len(ventas) == 675 and auditoria["duplicados_exactos"] == 5
assert np.isclose(ventas.Margen.sum(), ventas.Ingreso.sum() - ventas.Costo.sum())

,conteo
filas_originales,680
duplicados_exactos,5
ventas_unicas,675
canales_normalizados,30
clientes_sin_segmento,8
ventas_anteriores_al_alta,63


,IdVenta,Fecha,Canal,Ingreso,Margen,Alta_posterior
0,V00001,2026-01-01,Sitio web,262.2652,84.7852,False
1,V00002,2026-01-01,Tienda física,95.0000,33.3300,True
2,V00003,2026-01-01,Sitio web,44.6064,11.7064,False
3,V00004,2026-01-01,Tienda física,83.5176,24.0676,True
4,V00005,2026-01-02,Tienda física,79.0000,21.6800,True


<a id="calidad"></a>
## 2. Contrato, depuración y límites

La hoja `Ventas` contiene 680 filas y cinco repeticiones exactas. Se retiran esas repeticiones,
se normaliza el diccionario de canales y se exige relación muchos-a-uno al unir dimensiones.
Los ocho clientes sin segmento conservan la etiqueta «Sin información». No se imputan categorías
por moda. La región usada es la de la venta; la región de residencia es otro atributo.

Se detectan ventas anteriores a `FechaAlta`. La fecha podría representar migración al CRM y no
inicio de la relación: sin confirmación no corresponde corregirla. Esas ventas se conservan para
conciliar ingresos, pero no sirven para calcular antigüedad o adquisición. Compare el resultado
con y sin ellas como **sensibilidad**, sin presentar la exclusión como verdad.

In [3]:
contrato = pd.DataFrame(
    [
        ["IdVenta", "Clave única", ventas.IdVenta.is_unique],
        ["DescuentoPct", "Fracción entre 0 y 1", ventas.DescuentoPct.between(0, 1).all()],
        [
            "Unidades",
            "Entero positivo",
            (ventas.Unidades.gt(0) & ventas.Unidades.mod(1).eq(0)).all(),
        ],
        [
            "Ingreso",
            "Unidades × precio × (1-descuento)",
            np.allclose(
                ventas.Ingreso, ventas.Unidades * ventas.PrecioLista * (1 - ventas.DescuentoPct)
            ),
        ],
        ["FechaAlta", "Anterior o igual a venta", not ventas.Alta_posterior.any()],
    ],
    columns=["campo", "regla", "cumple"],
)
display(contrato)


@widgets.interact(
    excluir_altas=widgets.Checkbox(value=False, description="Excluir fechas conflictivas")
)
def sensibilidad_calidad(excluir_altas=False):
    f = ventas.loc[~ventas.Alta_posterior] if excluir_altas else ventas
    display(pd.Series(indicadores(f), name="resultado").to_frame())

,campo,regla,cumple
0,IdVenta,Clave única,True
1,DescuentoPct,Fracción entre 0 y 1,True
2,Unidades,Entero positivo,True
3,Ingreso,Unidades × precio × (1-descuento),True
4,FechaAlta,Anterior o igual a venta,False


interactive(children=(Checkbox(value=False, description='Excluir fechas conflictivas'), Output()), _dom_classe…

<a id="cohortes"></a>
## 3. BoldoNet: madurez no es abandono

La ficha fija el corte en **5 de agosto de 2026**. La antigüedad se calcula desde ingreso hasta ese
corte, no desde la fecha del equipo ni desde días de permanencia: un cliente que abandonó puede
tener permanencia corta y pertenecer a una cohorte madura. Se conserva la actualización más
reciente de cada cliente, tras retirar duplicados exactos; empates conflictivos detienen la carga.

Para retención a 90 días, el denominador observado incluye clientes con al menos 90 días de
ventana y estado válido. Debe informarse además cuántos maduros carecen de estado: excluirlos
sin reportarlo puede sesgar el KPI. Los inmaduros con estado se marcan para revisión; no entran
en esa tasa. El costo negativo se excluye del promedio de costo y se reporta por separado.

In [4]:
clientes, audit_clientes = boldonet()
display(pd.Series(audit_clientes, name="conteo"))
elegibles = clientes.loc[clientes.elegible_kpi]
maduros = clientes.loc[clientes.maduro]
activos = (elegibles["Estado_90_días"] == "Activo").sum()
desconocidos = (~maduros.estado_valido).sum()
tasa = activos / len(elegibles)
limites = (activos / len(maduros), (activos + desconocidos) / len(maduros))
display(
    pd.Series(
        {
            "maduros": len(maduros),
            "observados": len(elegibles),
            "retencion_observada": tasa,
            "limite_inferior_faltantes": limites[0],
            "limite_superior_faltantes": limites[1],
        }
    )
)
assert clientes.ID_cliente.is_unique and 0 <= limites[0] <= limites[1] <= 1

filas_originales                      246
clientes_unicos                       240
versiones_retiradas                     6
fechas_ingreso_invalidas                1
maduros_sin_estado_valido               6
no_maduros_con_estado                  18
costos_invalidos                        2
actualizaciones_anteriores_ingreso     21
actualizaciones_invalidas               1
satisfacciones_fuera_rango              4
satisfacciones_faltantes                6
permanencias_negativas                  2
elegibles_kpi                         121
elegibles_kpi_estricto                121
Name: conteo, dtype: int64

maduros                      127.000000
observados                   121.000000
retencion_observada            0.760331
limite_inferior_faltantes      0.724409
limite_superior_faltantes      0.771654
dtype: float64

### Fechas y sensibilidad de la retención

`Fecha_ingreso` se interpreta como inicio de la relación; `Fecha_actualización` como actualización de esa ficha.
Si la segunda precede a la primera, marcamos una contradicción y pedimos confirmar la semántica. No inventamos
una fecha. La tasa principal conserva las reglas de madurez/estado; la sensibilidad adicional excluye esos
casos y las actualizaciones desconocidas. Ambas se muestran con sus bases, sin presentar la exclusión como verdad.
La satisfacción fuera de 1–5 y permanencia negativa se excluyen de sus propios resúmenes, no se convierten en abandono.

In [5]:
filas_sensibilidad = []
for etiqueta, bandera in [
    ("Regla de madurez y estado", "elegible_kpi"),
    ("Además, cronología de ficha consistente", "elegible_kpi_estricto"),
]:
    grupo = clientes.loc[clientes[bandera]]
    filas_sensibilidad.append(
        {
            "regla": etiqueta,
            "n": len(grupo),
            "retencion": grupo["Estado_90_días"].eq("Activo").mean(),
        }
    )
display(pd.DataFrame(filas_sensibilidad))
display(
    clientes.loc[
        clientes.actualizacion_antes_ingreso,
        ["ID_cliente", "Fecha_ingreso", "Fecha_actualización", "maduro", "elegible_kpi"],
    ].head()
)
assert audit_clientes["actualizaciones_anteriores_ingreso"] == 21
assert audit_clientes["satisfacciones_fuera_rango"] == 4
assert audit_clientes["permanencias_negativas"] == 2

,regla,n,retencion
0,Regla de madurez y estado,121,0.760331
1,"Además, cronología de ficha consistente",121,0.760331


,ID_cliente,Fecha_ingreso,Fecha_actualización,maduro,elegible_kpi
50,CLI-0051,2026-07-12,2026-07-01,False,False
162,CLI-0159,2026-07-07,2026-07-01,False,False
5,CLI-0006,2026-07-28,2026-07-02,False,False
215,CLI-0211,2026-07-18,2026-07-02,False,False
242,CLI-0237,2026-07-19,2026-07-03,False,False


<a id="gobierno"></a>
## 4. Gobierno y uso responsable

Registre propietario del dato, finalidad, acceso, retención, diccionario y fecha de extracción.
Un dato disponible no necesariamente es pertinente. La ficha no entrega la composición total de
la cartera: no puede certificarse representatividad. Al comparar acompañamiento sí/no existen
posibles diferencias de riesgo previo y selección; una diferencia de retención no estima por sí
sola el efecto del programa. Proponga asignación aleatoria de un piloto entre elegibles, o un
diseño causal justificable cuando eso no sea posible.

**Ejercicio.** Diseñe una bitácora con la regla, filas afectadas, decisión, motivo y efecto sobre el
KPI. Establezca cuándo escalar una incidencia en vez de imputarla. **Solución orientadora:**
duplicado exacto → retirar con huella; canal conocido → normalizar; fecha de alta contradictoria →
marcar y consultar; estado faltante de maduro → informar rango; costo negativo → cuarentena
del cálculo de costo. Nunca escribir sobre `originales/`.

In [6]:
bitacora = pd.DataFrame(
    [
        [
            "CasaPeumo",
            "Duplicados exactos",
            auditoria["duplicados_exactos"],
            "Retirar copias en memoria",
        ],
        [
            "CasaPeumo",
            "Alta posterior",
            auditoria["ventas_anteriores_al_alta"],
            "Marcar; no usar antigüedad",
        ],
        [
            "BoldoNet",
            "Costos inválidos",
            audit_clientes["costos_invalidos"],
            "Excluir del promedio y reportar",
        ],
        [
            "BoldoNet",
            "Estado maduro desconocido",
            audit_clientes["maduros_sin_estado_valido"],
            "Informar límites",
        ],
    ],
    columns=["fuente", "regla", "afectados", "tratamiento"],
)
display(bitacora)

,fuente,regla,afectados,tratamiento
0,CasaPeumo,Duplicados exactos,5,Retirar copias en memoria
1,CasaPeumo,Alta posterior,63,Marcar; no usar antigüedad
2,BoldoNet,Costos inválidos,2,Excluir del promedio y reportar
3,BoldoNet,Estado maduro desconocido,6,Informar límites


<a id="autoevaluacion"></a>
## Autoevaluación
Seleccione y compruebe su respuesta.

In [7]:
auto_01 = pregunta(
    "¿Qué hacer con un cliente que todavía no completa 90 días?",
    [
        "Contarlo como abandono",
        "Excluirlo del KPI a 90 días e informar su cohorte",
        "Imputarlo como retenido",
    ],
    1,
    "No tuvo la misma ventana de observación. Informe madurez y faltantes antes de comparar tasas.",
)

## Práctica de transferencia 01

Hay 100 clientes maduros: 72 activos, 18 abandonos y 10 estados desconocidos; además 40 inmaduros. Calcula retención observada y límites por faltantes. Documenta una actualización previa al ingreso sin inventar fechas.

**Entrega:** hipótesis previa, cálculo con unidades, interpretación y límite. Completa tu respuesta antes de consultar la [pauta docente](../material_propio/PAUTA_DOCENTE.md#nb01).

**Discusión:** ¿Qué diferencia hay entre cobertura de estado y representatividad?

### Tu resolución

Edita esta celda: escribe tu hipótesis, procedimiento, resultado, interpretación y una comprobación. Adjunta tu código o gráfico si la actividad lo requiere.

**Auto-revisión:** ¿usé la población correcta?, ¿declaré unidades y supuestos?, ¿mi evidencia permite esa conclusión?